# Chapter 9 — Modeling Probabilities and Nonlinearities: Activation Functions

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning)
**Chapter 9:** Modeling Probabilities and Nonlinearities: Activation Functions

Notebook ini berisi reproduksi kode dari buku (diambil dari repository
resmi penulis) beserta penjelasan konsep dan rangkuman chapter.

## Pendahuluan

Chapter 9 membahas **activation function**: fungsi yang diterapkan pada
neuron untuk menghasilkan output (fungsi nonlinear).

### Apa itu activation function?

Fungsi yang diterapkan pada neuron dalam satu layer saat prediksi.
Fungsi ini harus memenuhi beberapa syarat: **kontinu dan tak terbatas
domainnya**, **monoton**, **nonlinear**, dan **efisien secara
komputasi** (derivatif mudah dihitung). Fungsi yang bersifat nonlinear
penting agar jaringan dapat mempelajari pola kompleks, sebab tanpa itu
semua layer setara dengan satu layer linear.

### Activation function standar pada hidden layer

- **Sigmoid:** output antara 0 dan 1 (cocok sebagai peluang). Turunan:
  `s * (1 - s)`.
- **Tanh:** output antara -1 dan 1. Pada hidden layer, tanh biasanya lebih
  baik daripada sigmoid karena nilai negatifnya membuat korelasi antar
  node dapat berkurang (lebih selektif). Turunan: `1 - t**2`.

### Activation function standar pada output layer

Tergantung target:
- Memprediksi **nilai mentah** (regresi): tanpa activation.
- Memprediksi **ya/tidak** independen: sigmoid.
- Memprediksi **satu dari banyak kelas**: **softmax**.

### Softmax

Masalah inti: input memiliki **kemiripan** antar kelas. Softmax menormalkan
semua output sehingga **berjumlah 1** (distribusi peluang) dengan
`exp(x) / sum(exp(x))`. Ini membuat jaringan menekan kelas lain ketika
satu kelas naik, sehingga lebih baik untuk klasifikasi satu label.

### Memasang activation ("installation instructions")

1. Terapkan activation pada layer saat **forward propagation**.
2. Saat backpropagation, **kalikan delta dengan turunan (slope)**
   activation: `delta = delta_dari_atas * turunan(output)`. Turunan
   dihitung dari **output** (bukan input) agar efisien, misal sigmoid:
   `output * (1 - output)`.

# Upgrading our MNIST Network

**Meningkatkan jaringan MNIST:** jaringan MNIST dari chapter sebelumnya
diganti dengan **tanh** pada hidden layer dan **softmax** pada output
layer, serta inisialisasi weights dan alpha yang disesuaikan. Hasilnya
akurasi uji meningkat dibanding versi sebelumnya karena model lebih
sesuai dengan sifat masalah (klasifikasi satu dari 10 kelas).

In [ ]:
import numpy as np, sys
np.random.seed(1)

from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000,28*28) / 255, y_train[0:1000])

one_hot_labels = np.zeros((len(labels),10))
for i,l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test.reshape(len(x_test),28*28) / 255
test_labels = np.zeros((len(y_test),10))
for i,l in enumerate(y_test):
    test_labels[i][l] = 1

def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

alpha, iterations, hidden_size = (2, 300, 100)
pixels_per_image, num_labels = (784, 10)
batch_size = 100

weights_0_1 = 0.02*np.random.random((pixels_per_image,hidden_size))-0.01
weights_1_2 = 0.2*np.random.random((hidden_size,num_labels)) - 0.1

for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end=((i * batch_size),((i+1)*batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_1 = tanh(np.dot(layer_0,weights_0_1))
        dropout_mask = np.random.randint(2,size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1,weights_1_2))

        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == np.argmax(labels[batch_start+k:batch_start+k+1]))

        layer_2_delta = (labels[batch_start:batch_end]-layer_2) / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    test_correct_cnt = 0

    for i in range(len(test_images)):

        layer_0 = test_images[i:i+1]
        layer_1 = tanh(np.dot(layer_0,weights_0_1))
        layer_2 = np.dot(layer_1,weights_1_2)

        test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))
    if(j % 10 == 0):
        sys.stdout.write("\n"+ \
         "I:" + str(j) + \
         " Test-Acc:"+str(test_correct_cnt/float(len(test_images)))+\
         " Train-Acc:" + str(correct_cnt/float(len(images))))

## Key Ideas

- Activation function memberi nonlinearitas pada neuron.
- Sigmoid untuk peluang (0-1), tanh untuk hidden layer, softmax untuk multi-kelas.
- Softmax membuat output berjumlah 1 sehingga kelas saling bersaing.
- Delta harus dikalikan turunan activation (slope) saat backpropagation.

## Kesimpulan


Chapter 9 menunjukkan bahwa pemilihan **activation function** yang tepat
meningkatkan kemampuan jaringan memodelkan peluang dan pola nonlinear:
**tanh** (atau ReLU) untuk hidden layer, **softmax** untuk klasifikasi
banyak kelas pada output. Saat backpropagation, delta harus dikalikan
dengan turunan activation yang dipakai pada layer tersebut.